# 📈 Sales Prediction & Advertising Analytics Project
### End-to-End Machine Learning Portfolio Project
**Objective**: Predict sales volume and analyze the efficiency of advertising expenditures across **TV**, **Radio**, and **Newspaper** channels using regression algorithms.

## 1. Import Required Libraries
We use standard data science and machine learning libraries in Python:

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (8, 5)
print('All libraries imported successfully!')

## 2. Load Dataset (`Advertising.csv`)
Let's load the dataset and inspect its structure, dimensions, and statistical properties.

In [ ]:
data_path = '../data/Advertising.csv' if os.path.exists('../data/Advertising.csv') else 'data/Advertising.csv'
df = pd.read_csv(data_path)

print('Dataset Shape:', df.shape)
df.head()

In [ ]:
# View bottom records
df.tail()

In [ ]:
# Check column data types and missing values
df.info()

In [ ]:
# Summary statistics
df.describe()

## 3. Data Cleaning
- Drop `Unnamed: 0` which represents row numbers from the CSV export.
- Check for missing values and duplicates.

In [ ]:
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
    print('Removed Unnamed: 0 index column')

print('Missing Values Check:')
print(df.isnull().sum())

print('\nDuplicate Rows Check:', df.duplicated().sum())
print('\nCleaned Data Preview:')
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's explore the relationships between advertising channels (**TV**, **Radio**, **Newspaper**) and **Sales**.

In [ ]:
# Correlation Matrix
corr = df.corr()
print('Correlation Matrix:')
print(corr)

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap='Blues', fmt='.3f', linewidths=1)
plt.title('Correlation Heatmap', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
# Scatter plots of each advertising channel against Sales
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
channels = [('TV', '#1d3557'), ('Radio', '#e63946'), ('Newspaper', '#2a9d8f')]

for i, (channel, color) in enumerate(channels):
    sns.regplot(data=df, x=channel, y='Sales', color=color, ax=axes[i], line_kws={'color': 'black', 'linewidth': 2})
    r_val = corr.loc[channel, 'Sales']
    axes[i].set_title(f'{channel} vs Sales (r = {r_val:.3f})', fontweight='bold')
    axes[i].set_xlabel(f'{channel} Ad Budget ($k)')
    if i == 0:
        axes[i].set_ylabel('Sales (k units)')

plt.suptitle('Advertising Expenditure vs Sales Across Channels', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Feature Selection & Train/Test Split
- **Features ($X$)**: TV, Radio, Newspaper
- **Target ($y$)**: Sales
- **Test Size**: 20% (40 samples), **Training Size**: 80% (160 samples)

In [ ]:
X = df[['TV', 'Radio', 'Newspaper']]
y = df['Sales']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training shape: {X_train.shape}')
print(f'Testing shape : {X_test.shape}')

## 6. Build and Evaluate Multiple Regression Models
We train and benchmark 4 algorithms:
1. **Linear Regression** (Baseline parametric model)
2. **Decision Tree Regressor** (Non-linear decision splits)
3. **Random Forest Regressor** (Ensemble bagging)
4. **Gradient Boosting Regressor** (Ensemble sequential boosting)

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2
    })

results_df = pd.DataFrame(results).sort_values(by='R2', ascending=False).reset_index(drop=True)
results_df

In [ ]:
# Visualize Model Comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(data=results_df, x='Model', y='R2', palette='Blues_r', ax=axes[0])
axes[0].set_title('R² Score (Higher is Better)', fontweight='bold')
axes[0].set_ylim(0, 1.05)

sns.barplot(data=results_df, x='Model', y='RMSE', palette='Reds', ax=axes[1])
axes[1].set_title('RMSE (Lower is Better)', fontweight='bold')

for ax in axes:
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

## 7. Actual vs Predicted Analysis (Best Model)
Let's inspect test predictions against actual ground-truth sales.

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]
y_pred_best = best_model.predict(X_test)

comp_table = pd.DataFrame({
    'TV Spend': X_test['TV'].values,
    'Radio Spend': X_test['Radio'].values,
    'Newspaper Spend': X_test['Newspaper'].values,
    'Actual Sales': y_test.values,
    'Predicted Sales': np.round(y_pred_best, 2),
    'Error': np.round(np.abs(y_test.values - y_pred_best), 2)
})

print(f'Top 10 Predictions with {best_model_name}:')
comp_table.head(10)

In [ ]:
# Actual vs. Predicted Scatter Plot
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred_best, color='#1d3557', alpha=0.8, edgecolors='k', s=50, label='Predictions')
min_val = min(y_test.min(), y_pred_best.min()) - 1
max_val = max(y_test.max(), y_pred_best.max()) + 1
plt.plot([min_val, max_val], [min_val, max_val], color='#e63946', linestyle='--', linewidth=2, label='Ideal Fit')
plt.title(f'Actual vs. Predicted Sales ({best_model_name})', fontweight='bold')
plt.xlabel('Actual Sales (k units)')
plt.ylabel('Predicted Sales (k units)')
plt.legend()
plt.show()

## 8. Feature Importance & Coefficients Analysis

In [ ]:
# Linear Regression Coefficients
lr = models['Linear Regression']
print('Linear Regression Formula:')
print(f'Sales = {lr.intercept_:.4f} + ({lr.coef_[0]:.4f} * TV) + ({lr.coef_[1]:.4f} * Radio) + ({lr.coef_[2]:.4f} * Newspaper)')

# Ensemble Feature Importance
if hasattr(best_model, 'feature_importances_'):
    fi_df = pd.DataFrame({
        'Channel': ['TV', 'Radio', 'Newspaper'],
        'Importance': best_model.feature_importances_
    }).sort_values(by='Importance', ascending=False)
    
    plt.figure(figsize=(6, 3))
    sns.barplot(data=fi_df, x='Importance', y='Channel', palette='viridis')
    plt.title(f'Feature Importance ({best_model_name})', fontweight='bold')
    plt.show()

## 9. Marketing Scenario & What-If Simulator

In [ ]:
def simulate_budget(tv, radio, newspaper):
    pred = best_model.predict(pd.DataFrame([{'TV': tv, 'Radio': radio, 'Newspaper': newspaper}]))[0]
    return round(float(pred), 2)

# Scenario Comparison
scenario_a_pred = simulate_budget(100, 20, 10)
scenario_b_pred = simulate_budget(150, 30, 20)

print(f'Scenario A (TV=100, Radio=20, Newspaper=10): {scenario_a_pred:.2f}k units')
print(f'Scenario B (TV=150, Radio=30, Newspaper=20): {scenario_b_pred:.2f}k units')
print(f'Difference: {scenario_b_pred - scenario_a_pred:+.2f}k units ({((scenario_b_pred - scenario_a_pred)/scenario_a_pred)*100:+.1f}%)')

## 10. Save Model Artifact for Deployment

In [ ]:
os.makedirs('../models', exist_ok=True)
artifact = {
    'model': best_model,
    'model_name': best_model_name,
    'feature_names': ['TV', 'Radio', 'Newspaper'],
    'metrics': results_df.iloc[0].to_dict()
}
joblib.dump(artifact, '../models/sales_prediction_model.pkl')
print('Model artifact saved successfully!')